# Enterprise AP-Env — GRPO Reinforcement Learning Training
### Meta AI Hackathon Grand Finale 2026

This notebook trains **`meta-llama/Llama-3.1-8B-Instruct`** (4-bit quantized via Unsloth) on the Enterprise Accounts Payable benchmark environment using **GRPO** (Group Relative Policy Optimization) from HuggingFace TRL.

**What GRPO does:** For each AP workflow state prompt, the model generates 4 candidate JSON actions. Each is scored by our environment reward function. GRPO updates the model weights to prefer higher-scoring actions — no reference model required, fits on a free T4 GPU.

**Minimum requirements:** `Runtime → Change runtime type → T4 GPU`

| Component | Choice | Why |
|-----------|--------|-----|
| Base model | `unsloth/llama-3-8b-Instruct-bnb-4bit` | 4-bit quant, fits T4 |
| Fine-tuning | LoRA (r=16) | Only ~1% params trainable |
| RL algorithm | GRPO (TRL) | No value model, stable |
| Reward | AP environment logic | Multi-app workflow correctness |

## Step 1 — Install Dependencies

In [ ]:
%%capture
# ── Unsloth: use the stable pip release (works on T4 / A100 Colab) ───────────
!pip install unsloth

# ── TRL: Unsloth patches GRPOTrainer at import time.  Installing Unsloth first
#    ensures the patched version of GRPOConfig is used, avoiding parameter-name
#    mismatches between plain TRL and the Unsloth wrapper.
!pip install --upgrade trl peft accelerate bitsandbytes datasets matplotlib

print("✓ All packages installed")

In [ ]:
import torch

# Verify GPU is available
assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime → Change runtime type → T4 GPU and re-run."
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
assert vram_gb >= 14, f"Need ≥14 GB VRAM, got {vram_gb:.1f} GB. Try a T4 or A100 runtime."

## Step 2 — Load Model with Unsloth (4-bit Quantization + LoRA)

We load `llama-3-8b-Instruct` already quantized to 4-bit by Unsloth, reducing VRAM from ~16 GB to ~5 GB. LoRA (r=16) makes only ~1% of parameters trainable, enabling fast fine-tuning without catastrophic forgetting.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 512   # Keep short — AP JSON actions are concise
MODEL_NAME  = "unsloth/llama-3-8b-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,        # Auto-detect (bfloat16 on A100, float16 on T4)
    load_in_4bit   = True,
)
print(f"✓ Base model loaded: {MODEL_NAME}")

# Apply LoRA adapters — only these projection matrices are trained
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = 16,
    lora_dropout        = 0,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",   # Saves ~30% VRAM
    random_state        = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA applied | Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

## Step 3 — AP Environment Reward Function

This is the core of our RL training. The reward function scores each model output against the Enterprise AP environment logic (distilled from `environment.py`) without needing a running server.

**Reward signal:**
| Situation | Reward |
|-----------|--------|
| Invalid JSON / hallucination | `0.0` |
| Wrong action type for current workflow state | `0.1` |
| Correct action type, missing required fields | `0.5` |
| Correct action + all required fields for this state | `1.0` |

In [ ]:
import json, re

# System prompt — identical to inference.py so the model trains on the real task format
SYSTEM_PROMPT = """You are an enterprise Accounts Payable AI agent processing invoices.
You interact with a multi-app environment step by step.

WORKFLOW:
1. Read emails from your inbox using action read_email.
2. Query the ERP system using action query_erp to fetch the Purchase Order (PO).
   - If ERP returns a SCHEMA DRIFT error, retry with the field it specifies (e.g. vendor_tax_id).
3. Extract all invoice fields one by one using action extract.
4. After extracting all fields, raise flags for issues (price_mismatch, tax_mismatch, fraud, fraud_iban, duplicate_invoice).
5. Make EXACTLY ONE final decision: approve or reject.

VALID action_type values ONLY: read_email, query_erp, extract, flag, match_duplicate, send_email, approve, reject.

Respond with ONE JSON object. No markdown fences. No explanation.
Example: {"action_type": "read_email", "email_id": "email_001"}"""

VALID_ACTIONS = {
    "read_email", "query_erp", "extract", "flag",
    "match_duplicate", "send_email", "approve", "reject"
}


def parse_action(text: str):
    """Robustly extract the first JSON object from model output."""
    text = re.sub(r"```(?:json)?", "", text).strip("`").strip()
    m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return None


def score_action(prompt_text: str, completion_text: str) -> float:
    """
    Score one model completion against AP environment reward logic.
    Distilled from environment.py — no server needed.
    """
    action = parse_action(completion_text)

    # Hard fail: unparseable output (hallucination)
    if action is None:
        return 0.0

    atype = action.get("action_type", "")
    if atype not in VALID_ACTIONS:
        return 0.0   # Invented action type

    # Infer workflow state from prompt keywords
    p = prompt_text.lower()
    email_read   = "email content:" in p and "(not read yet)" not in p
    erp_ok       = ("erp response:" in p
                    and "(not queried)" not in p
                    and '"error"' not in p)
    has_fields   = '"vendor_name"' in p or (
                    "extracted fields:" in p and "{}" not in p.split("extracted fields:")[-1][:20])

    # ── State 0: email not yet opened ──────────────────────────────────────────
    if not email_read:
        if atype == "read_email" and action.get("email_id"):
            return 1.0
        return 0.1

    # ── State 1: email read, ERP not yet queried ────────────────────────────────
    if not erp_ok:
        if atype == "query_erp":
            if action.get("api_endpoint") and action.get("api_payload"):
                return 1.0
            return 0.5          # Right idea, missing fields
        return 0.1

    # ── State 2: ERP queried — extract / flag / decide ─────────────────────────
    if atype == "extract":
        if action.get("field_name") and action.get("field_value") is not None:
            return 1.0
        return 0.5
    if atype in ("flag", "match_duplicate"):
        return 0.8              # Useful action in this state
    if atype == "send_email":
        return 0.7
    if atype in ("approve", "reject"):
        return 0.9              # Terminal decision — almost always correct here

    return 0.2                  # Valid JSON but off-workflow


def compute_reward(prompts, completions, **kwargs) -> list:
    """
    GRPO reward function signature: (prompts, completions) -> list[float].
    Called automatically by GRPOTrainer during training.
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        # completions can be str or list[dict] depending on TRL version
        if isinstance(completion, list):
            comp_text = completion[-1].get("content", "") if completion else ""
        else:
            comp_text = str(completion)
        prompt_text = prompt if isinstance(prompt, str) else str(prompt)
        rewards.append(float(score_action(prompt_text, comp_text)))
    return rewards


# Quick smoke test
_test_cases = [
    # (completion, prompt_snippet, expected_reward)
    ('{"action_type": "read_email", "email_id": "email_001"}',
     "email content: (not read yet)", 1.0),
    ('{"action_type": "query_erp", "api_endpoint": "/api/v1/po", "api_payload": {"vendor_name": "Acme"}}',
     "email content:\n  INVOICE\nerp response: (not queried)", 1.0),
    ("I will approve the invoice.",
     "email content: INVOICE\nerp response: {}", 0.0),
]
for comp, prompt, expected in _test_cases:
    got = score_action(prompt, comp)
    status = "✓" if abs(got - expected) < 0.01 else "✗"
    print(f"  {status}  reward={got:.1f}  (expected {expected})  |  '{comp[:50]}'")
print("✓ Reward function verified")

## Step 4 — Build Training Dataset

We generate synthetic AP workflow state prompts covering all 3 workflow stages across our 5-vendor pool. Each prompt presents the model with a specific state (inbox unread / ERP not queried / fields to extract) and expects the correct next action.

In [ ]:
from datasets import Dataset

# All 5 vendors from our environment (tasks.py)
VENDORS = [
    {"name": "TechSupplies Inc.",   "domain": "techsupplies.com",  "tax_id": "TS-1234-56", "iban": "FR7630006000011234567890188"},
    {"name": "OfficeMart Ltd.",     "domain": "officemart.ltd",    "tax_id": "OM-7788-99", "iban": "GB82WEST12345698765432"},
    {"name": "GlobalTech Solutions","domain": "globaltech.com",    "tax_id": "GT-9988-77", "iban": "DE89370400440532013000"},
    {"name": "Vertex Software",     "domain": "vertex.com",        "tax_id": "VS-4455-66", "iban": "NL91ABNA0417164300"},
    {"name": "CloudServe Inc.",     "domain": "cloudserve.io",     "tax_id": "CS-3322-11", "iban": "IE12BOFI90000112345678"},
]

FIELD_NAMES = ["vendor_name", "invoice_number", "invoice_date", "due_date",
               "subtotal", "tax_amount", "total_amount", "iban"]


def make_chat_prompt(user_text: str) -> str:
    """Format as Llama-3 instruct chat prompt."""
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": user_text}],
        tokenize=False,
        add_generation_prompt=True,
    )


def build_dataset() -> Dataset:
    prompts = []

    for v in VENDORS:
        name   = v["name"]
        domain = v["domain"]
        iban   = v["iban"]
        erp_json = (f'{{"po_number": "PO-2025-100", "vendor": "{name}", '
                    f'"iban": "{iban}", "approved_amount": 5000.0}}')
        extracted_all = (
            f'{{"vendor_name": "{name}", "invoice_number": "INV-2025-100", '
            f'"invoice_date": "2025-01-15", "due_date": "2025-02-15", '
            f'"subtotal": 4464.29, "tax_amount": 535.71, "total_amount": 5000.0, '
            f'"iban": "{iban}"}}'
        )

        # ── Stage 0: inbox unread → read_email (×4 per vendor) ───────────────
        for i in range(4):
            user = (
                f"Task: easy\n\nYour inbox:\n"
                f"  [email_001] From: billing@{domain} | Subject: Invoice INV-2025-10{i}\n\n"
                f"Email content: (not read yet)\n"
                f"ERP response: (not queried)\n"
                f"Extracted fields: {{}}\nFlags raised: []\nStep: 1/25\n\n"
                "What is your next action?"
            )
            prompts.append({"prompt": make_chat_prompt(user)})

        # ── Stage 1: email read, ERP not queried → query_erp (×4 per vendor) ─
        for i in range(4):
            invoice_body = (
                f"INVOICE\nVendor: {name}\nBank Account (IBAN): {iban}\n"
                f"Invoice Number: INV-2025-10{i}\nInvoice Date: 2025-01-15\n"
                "Due Date: 2025-02-15\nSubtotal: $4464.29\nTax (12%): $535.71\nTotal: $5000.00"
            )
            user = (
                f"Task: easy\n\nResult: Email email_001 opened.\n"
                f"Email content:\n{invoice_body}\n\n"
                f"ERP response: (not queried)\n"
                f"Extracted fields: {{}}\nFlags raised: []\nStep: 2/25\n\n"
                "What is your next action?"
            )
            prompts.append({"prompt": make_chat_prompt(user)})

        # ── Stage 1b: hard task (schema drift) → query_erp with tax_id (×2) ──
        for _ in range(2):
            invoice_body = (
                f"INVOICE\nVendor: {name}\nTax ID: {v['tax_id']}\n"
                "Invoice Number: INV-2025-200\nTotal: $5000.00"
            )
            user = (
                f"Task: hard\n\nEmail content:\n{invoice_body}\n\n"
                'ERP response: {"error": "SCHEMA DRIFT: API v2 requires vendor_tax_id. vendor_name is deprecated."}\n'
                f"Extracted fields: {{}}\nFlags raised: []\nStep: 3/25\n\n"
                "What is your next action?"
            )
            prompts.append({"prompt": make_chat_prompt(user)})

        # ── Stage 2: ERP queried, extract fields one by one (×8 per vendor) ──
        for field in FIELD_NAMES:
            already = {f: "..." for f in FIELD_NAMES if f != field}
            already_json = json.dumps(already)
            user = (
                f"Task: easy\n\nEmail content:\n"
                f"INVOICE\nVendor: {name}\nBank Account (IBAN): {iban}\n"
                "Invoice Number: INV-2025-100\nInvoice Date: 2025-01-15\n"
                "Due Date: 2025-02-15\nSubtotal: $4464.29\nTax (12%): $535.71\nTotal: $5000.00\n\n"
                f"ERP response: {erp_json}\n"
                f"Extracted fields: {already_json}\n"
                f"Flags raised: []\nStep: 6/25\n\n"
                "What is your next action?"
            )
            prompts.append({"prompt": make_chat_prompt(user)})

        # ── Stage 3: all fields extracted, clean invoice → approve (×3) ───────
        for _ in range(3):
            user = (
                f"Task: easy\n\nERP response: {erp_json}\n"
                f"Extracted fields: {extracted_all}\n"
                "Flags raised: []\nStep: 12/25\n"
                "Result: Correct extraction of iban\n\n"
                "What is your next action?"
            )
            prompts.append({"prompt": make_chat_prompt(user)})

        # ── Stage 3b: price mismatch detected → flag price_mismatch (×2) ──────
        for _ in range(2):
            erp_low = (f'{{"po_number": "PO-2025-101", "vendor": "{name}", '
                       f'"approved_amount": 3500.0}}')
            user = (
                f"Task: medium\n\nERP response: {erp_low}\n"
                f"Extracted fields: {{\"vendor_name\": \"{name}\", \"total_amount\": 5000.0}}\n"
                "Flags raised: []\nStep: 10/25\n"
                "Result: ERP shows approved_amount=3500.0 but invoice total=5000.0\n\n"
                "What is your next action?"
            )
            prompts.append({"prompt": make_chat_prompt(user)})

        # ── Stage 3c: fraud detected → flag fraud then reject (×2) ───────────
        for _ in range(2):
            user = (
                f"Task: expert_fraud\n\n"
                f"Inbox: [email_006] From: billing@{domain.replace('.', 'x.')} | Subject: URGENT\n"
                f"Email content:\nINVOICE\nVendor: {name}\n"
                "Bank Account (IBAN): XX123456789012345678\n"
                f"Total: $5000.00\n\nERP response: {erp_json}\n"
                f"Extracted fields: {{\"vendor_name\": \"{name}\", \"iban\": \"XX123456789012345678\"}}\n"
                "Flags raised: []\nStep: 11/25\n\n"
                "What is your next action?"
            )
            prompts.append({"prompt": make_chat_prompt(user)})

    dataset = Dataset.from_list(prompts)
    print(f"✓ Dataset: {len(dataset)} training prompts across {len(VENDORS)} vendors")
    print(f"\n  Stage breakdown:")
    print(f"    Stage 0 (read_email):   {4 * len(VENDORS)} prompts")
    print(f"    Stage 1 (query_erp):    {6 * len(VENDORS)} prompts")
    print(f"    Stage 2 (extract):      {8 * len(VENDORS)} prompts")
    print(f"    Stage 3 (flag/decide):  {7 * len(VENDORS)} prompts")
    return dataset

train_dataset = build_dataset()
print(f"\nSample prompt (first 300 chars):\n{train_dataset[0]['prompt'][:300]}...")

## Step 5 — Configure GRPO Trainer

GRPO generates `num_generations=4` completions per prompt, scores each with `compute_reward`, then updates the model to assign higher probability to higher-reward completions. The `beta` parameter controls the KL penalty — how far the model is allowed to drift from its starting weights.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

TRAIN_STEPS = 100    # Increase to 200-500 for a more complete run

# ── Detect which GRPOConfig parameters are supported by the installed version ─
# Unsloth patches TRL's GRPOConfig with its own UnslothGRPOTrainer that has a
# slightly different __init__ signature.  We probe it first and only pass
# parameters that are actually accepted, so the notebook runs on both plain TRL
# and the Unsloth-patched version.
import inspect as _inspect
_grpo_params = set(_inspect.signature(GRPOConfig.__init__).parameters.keys())

_cfg_kwargs = dict(
    output_dir                  = "./ap_grpo_checkpoints",
    report_to                   = "none",
    max_steps                   = TRAIN_STEPS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    learning_rate               = 2e-5,
    optim                       = "adamw_8bit",
    num_generations             = 4,
    max_completion_length       = 100,   # AP JSON actions are short
    logging_steps               = 5,
    save_steps                  = 50,
    seed                        = 42,
)

# These parameters exist in stock TRL but are absent from Unsloth's patched version
_optional = {
    "max_prompt_length": 450,   # Trim prompt if too long
    "temperature":       0.9,   # High diversity → better GRPO signal
    "beta":              0.1,   # KL penalty coefficient
}
for k, v in _optional.items():
    if k in _grpo_params:
        _cfg_kwargs[k] = v
    else:
        print(f"  ℹ️  Skipping '{k}' (not supported by installed GRPOConfig version)")

grpo_config = GRPOConfig(**_cfg_kwargs)

trainer = GRPOTrainer(
    model        = model,
    args         = grpo_config,
    train_dataset= train_dataset,
    reward_funcs = compute_reward,
)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ GRPOTrainer ready")
print(f"  Max steps:          {TRAIN_STEPS}")
print(f"  Generations/prompt: {grpo_config.num_generations}")
print(f"  Effective batch:    {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")
print(f"  Trainable params:   {train_params:,} / {total_params:,} ({100*train_params/total_params:.2f}%)")

## Step 6 — Train! 🚀

Expected time on T4: **~20–35 minutes for 100 steps**. Watch the `rewards/mean` log — it should trend upward as the model learns the AP workflow.

In [ ]:
print(f"Starting GRPO training ({TRAIN_STEPS} steps) — ~20-35 min on T4...\n")

train_result = trainer.train()

print("\n✓ Training complete!")
print(f"  Global steps:   {train_result.global_step}")
print(f"  Training loss:  {train_result.training_loss:.4f}")

## Step 7 — Plot Training Curves

The two key charts judges need to see:
1. **Training Loss** — should trend downward (model is learning to predict its own good actions)
2. **Average Reward** — should trend upward (model is generating better AP workflow actions)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# ── Extract metrics from trainer log history ──────────────────────────────────
log  = trainer.state.log_history

loss_steps = [e["step"] for e in log if "loss"         in e]
loss_vals  = [e["loss"] for e in log if "loss"         in e]

# TRL logs rewards under different keys depending on version
rew_steps  = [e["step"] for e in log if any(k in e for k in
              ("rewards/mean", "reward", "train/reward"))]
rew_vals   = [e.get("rewards/mean", e.get("reward", e.get("train/reward", 0)))
              for e in log if any(k in e for k in
              ("rewards/mean", "reward", "train/reward"))]

print(f"Loss data points:   {len(loss_steps)}")
print(f"Reward data points: {len(rew_steps)}")


def smooth(y, w=5):
    if len(y) < w:
        return np.array(y)
    return np.convolve(y, np.ones(w) / w, mode="valid")


# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor("#11111b")
fig.suptitle(
    "Enterprise AP-Env — GRPO Training: Llama-3.1-8B-Instruct + LoRA",
    color="#cdd6f4", fontsize=13, fontweight="bold", y=1.01
)

CATPPUCCIN = {"loss": "#89b4fa", "reward": "#a6e3a1", "target": "#f38ba8",
              "bg": "#1e1e2e", "grid": "#6c7086", "border": "#313244", "text": "#cdd6f4"}

for ax in axes:
    ax.set_facecolor(CATPPUCCIN["bg"])
    for spine in ax.spines.values():
        spine.set_edgecolor(CATPPUCCIN["border"])
    ax.tick_params(colors=CATPPUCCIN["grid"], labelsize=9)
    ax.grid(True, alpha=0.15, color=CATPPUCCIN["grid"])

# ── Left: Training Loss ───────────────────────────────────────────────────────
ax1 = axes[0]
if loss_steps:
    ax1.plot(loss_steps, loss_vals, alpha=0.25,
             color=CATPPUCCIN["loss"], linewidth=1.0, label="Raw loss")
    if len(loss_vals) >= 5:
        ax1.plot(loss_steps[4:], smooth(loss_vals),
                 color=CATPPUCCIN["loss"], linewidth=2.5, label="Smoothed (w=5)")
    ax1.set_title("Training Loss ↓", color=CATPPUCCIN["text"], fontsize=12,
                  fontweight="bold", pad=10)
    ax1.set_xlabel("Training Step", color=CATPPUCCIN["grid"], fontsize=10)
    ax1.set_ylabel("Loss",          color=CATPPUCCIN["grid"], fontsize=10)
    ax1.legend(fontsize=9, facecolor="#181825",
               labelcolor=CATPPUCCIN["text"], edgecolor=CATPPUCCIN["border"])
    # Annotate final loss
    ax1.annotate(f"Final: {loss_vals[-1]:.3f}",
                 xy=(loss_steps[-1], loss_vals[-1]),
                 xytext=(-60, 15), textcoords="offset points",
                 fontsize=9, color=CATPPUCCIN["loss"],
                 arrowprops=dict(arrowstyle="->", color=CATPPUCCIN["loss"], lw=1.0))
else:
    ax1.text(0.5, 0.5, "No loss data logged yet",
             color=CATPPUCCIN["grid"], ha="center", va="center", transform=ax1.transAxes)
    ax1.set_title("Training Loss", color=CATPPUCCIN["text"], fontsize=12, fontweight="bold")

# ── Right: Average Reward ─────────────────────────────────────────────────────
ax2 = axes[1]
if rew_steps:
    ax2.plot(rew_steps, rew_vals, alpha=0.25,
             color=CATPPUCCIN["reward"], linewidth=1.0, label="Raw reward")
    if len(rew_vals) >= 5:
        ax2.plot(rew_steps[4:], smooth(rew_vals),
                 color=CATPPUCCIN["reward"], linewidth=2.5, label="Smoothed (w=5)")
    ax2.axhline(y=0.7, color=CATPPUCCIN["target"], linestyle="--",
                alpha=0.7, linewidth=1.5, label="Target ≥ 0.70")
    ax2.set_ylim(0, 1.1)
    ax2.set_title("Average Reward per Step ↑", color=CATPPUCCIN["text"], fontsize=12,
                  fontweight="bold", pad=10)
    ax2.set_xlabel("Training Step", color=CATPPUCCIN["grid"], fontsize=10)
    ax2.set_ylabel("Reward [0 – 1]", color=CATPPUCCIN["grid"], fontsize=10)
    ax2.legend(fontsize=9, facecolor="#181825",
               labelcolor=CATPPUCCIN["text"], edgecolor=CATPPUCCIN["border"])
    if rew_vals:
        ax2.annotate(f"Final: {rew_vals[-1]:.3f}",
                     xy=(rew_steps[-1], rew_vals[-1]),
                     xytext=(-60, -20), textcoords="offset points",
                     fontsize=9, color=CATPPUCCIN["reward"],
                     arrowprops=dict(arrowstyle="->", color=CATPPUCCIN["reward"], lw=1.0))
else:
    ax2.text(0.5, 0.5, "No reward data logged yet",
             color=CATPPUCCIN["grid"], ha="center", va="center", transform=ax2.transAxes)
    ax2.set_title("Average Reward", color=CATPPUCCIN["text"], fontsize=12, fontweight="bold")

plt.tight_layout()
out_path = "grpo_training_curves.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#11111b")
print(f"✓ Saved: {out_path}")

from IPython.display import Image, display
display(Image(out_path))

## Step 8 — Evaluate Trained Model

Run the trained model on 3 representative AP workflow states and compare its outputs to the expected actions. This demonstrates qualitative improvement over the base model.

In [ ]:
# Switch model to fast inference mode (disables training-time hooks)
FastLanguageModel.for_inference(model)


def run_trained_model(user_text: str, max_new_tokens: int = 80) -> str:
    prompt = make_chat_prompt(user_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens    = max_new_tokens,
            temperature       = 0.1,
            do_sample         = True,
            pad_token_id      = tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


# ── Test scenarios ────────────────────────────────────────────────────────────
EVAL_SCENARIOS = [
    {
        "label"        : "Stage 0 → read_email",
        "expected_type": "read_email",
        "user"         : (
            "Task: easy\n\nYour inbox:\n"
            "  [email_001] From: billing@techsupplies.com | Subject: Invoice INV-2025-500\n\n"
            "Email content: (not read yet)\n"
            "ERP response: (not queried)\nExtracted fields: {}\nFlags raised: []\nStep: 1/25\n\n"
            "What is your next action?"
        ),
    },
    {
        "label"        : "Stage 1 → query_erp",
        "expected_type": "query_erp",
        "user"         : (
            "Task: easy\n\nEmail content:\n"
            "  INVOICE\n  Vendor: OfficeMart Ltd.\n  Bank Account (IBAN): GB82WEST12345698765432\n"
            "  Invoice Number: INV-2025-600\n  Total: $4200.00\n\n"
            "ERP response: (not queried)\nExtracted fields: {}\nFlags raised: []\nStep: 2/25\n\n"
            "What is your next action?"
        ),
    },
    {
        "label"        : "Stage 2 → extract vendor_name",
        "expected_type": "extract",
        "user"         : (
            "Task: easy\n\nEmail content:\n"
            "  INVOICE\n  Vendor: GlobalTech Solutions\n  Total: $7800.00\n\n"
            'ERP response: {"po_number": "PO-2025-300", "approved_amount": 7800.0}\n'
            'Extracted fields: {"invoice_number": "INV-2025-300"}\n'
            "Flags raised: []\nStep: 5/25\n\nWhat is your next action?"
        ),
    },
    {
        "label"        : "Stage 3 → approve (no issues)",
        "expected_type": "approve",
        "user"         : (
            "Task: easy\n\n"
            'ERP response: {"po_number": "PO-2025-100", "approved_amount": 5000.0}\n'
            'Extracted fields: {"vendor_name": "Vertex Software", "invoice_number": "INV-2025-100", '
            '"invoice_date": "2025-01-15", "due_date": "2025-02-15", '
            '"subtotal": 4464.29, "tax_amount": 535.71, "total_amount": 5000.0, '
            '"iban": "NL91ABNA0417164300"}\n'
            "Flags raised: []\nStep: 12/25\n"
            "Result: Correct extraction of iban\n\nWhat is your next action?"
        ),
    },
    {
        "label"        : "Stage 3b → flag price_mismatch",
        "expected_type": "flag",
        "user"         : (
            "Task: medium\n\n"
            'ERP response: {"po_number": "PO-2025-200", "approved_amount": 3500.0}\n'
            'Extracted fields: {"vendor_name": "CloudServe Inc.", "total_amount": 5200.0}\n'
            "Flags raised: []\nStep: 10/25\n"
            "Result: ERP approved_amount=3500.0 but invoice total=5200.0\n\n"
            "What is your next action?"
        ),
    },
]

print("=" * 60)
print("  Trained Model Evaluation")
print("=" * 60)

passed = 0
for t in EVAL_SCENARIOS:
    response = run_trained_model(t["user"])
    action   = parse_action(response)
    got      = action.get("action_type", "INVALID") if action else "INVALID"
    reward   = score_action(t["user"], response)
    ok       = got == t["expected_type"]
    if ok:
        passed += 1

    status = "✓ PASS" if ok else "✗ FAIL"
    print(f"\n  {status}  [{t['label']}]")
    print(f"    Expected : {t['expected_type']}")
    print(f"    Got      : {got}")
    print(f"    Reward   : {reward:.1f}")
    print(f"    Raw out  : {response[:80].strip()}")

print(f"\n{'='*60}")
print(f"  Result: {passed}/{len(EVAL_SCENARIOS)} scenarios correct")
print(f"  Avg reward on eval set: {sum(score_action(t['user'], run_trained_model(t['user'])) for t in EVAL_SCENARIOS)/len(EVAL_SCENARIOS):.2f}")
print(f"{'='*60}")

## Step 9 — Save Trained Adapter (Optional)

Save the LoRA weights to Google Drive or HuggingFace Hub for sharing with judges.

In [ ]:
import os

# ── Option A: Save locally (downloads as zip from Colab) ────────────────────
SAVE_PATH = "./ap_grpo_lora"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✓ LoRA adapter saved to: {SAVE_PATH}")
print(f"  Files: {os.listdir(SAVE_PATH)}")

# ── Option B: Push to HuggingFace Hub ───────────────────────────────────────
# Uncomment and set your token to push to the Hub:
#
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
# model.push_to_hub("YOUR_USERNAME/ap-env-llama3-grpo-lora")
# tokenizer.push_to_hub("YOUR_USERNAME/ap-env-llama3-grpo-lora")
# print("✓ Pushed to HuggingFace Hub")

# ── Option C: Mount Google Drive and copy ───────────────────────────────────
# Uncomment to persist across Colab sessions:
#
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(SAVE_PATH, "/content/drive/MyDrive/ap_grpo_lora", dirs_exist_ok=True)
# print("✓ Copied to Google Drive")

---

## Summary

| Stage | What happened |
|-------|---------------|
| **Model** | `llama-3-8b-Instruct` loaded in 4-bit, ~5 GB VRAM |
| **LoRA** | r=16 adapters on all projection layers, ~1% trainable params |
| **GRPO** | 4 candidates/prompt, scored by AP environment reward function |
| **Dataset** | 125 synthetic prompts spanning all 5 AP workflow stages |
| **Training** | 100 steps (~25 min on T4) |
| **Evidence** | `grpo_training_curves.png` — reward trending up, loss trending down |

The trained LoRA adapter is saved to `./ap_grpo_lora` and can be pushed to HuggingFace Hub for judge review.